In [1]:
import pandas as pd
from pathlib import Path
from harbor.analysis.cross_docking import (
    DataFrameModel,
    DataFrameType,
    DockingDataModel,
)
from harbor.analysis.utils import FileLogger
import click
import numpy as np
import json

In [2]:
pose_data = list(Path("/data1/choderaj/paynea/asap-datasets/full_cross_dock_v2/docked_files/FRED_1_poses").glob("*_docked/*.csv"))

In [3]:
len(pose_data)

498

In [4]:
pose_df = pd.concat([pd.read_csv(csv) for csv in pose_data])

In [5]:
pose_df.columns

Index(['SMILES', 'target_identifiers', 'ligand_id', 'input',
       'docking-confidence-POSIT', 'INCHIKEY', 'complex_ligand_smiles',
       'pose_id', 'docking-structure-POSIT', 'ligand_identifiers',
       'docking-score-POSIT'],
      dtype='object')

In [6]:
pose_df

,SMILES,target_identifiers,ligand_id,input,docking-confidence-POSIT,INCHIKEY,complex_ligand_smiles,pose_id,docking-structure-POSIT,ligand_identifiers,docking-score-POSIT
0,Cc1ccncc1NC(=O)[C@@H]2CCC3(C2)CC3,NaN,MAT-POS-590ac91e-21,type='POSITDockingResults' input_pair=DockingI...,0.36,FNCISWRYQRGLKF-LLVKDONJSA-N,Cn1c2c(cncc2NC(=O)Cc3cccc(c3)Cl)cn1,0,Mpro-P1470_0A,NaN,-6.400138
1,Cc1ccncc1NC(=O)[C@@H]2CCC3(C2)CC3,NaN,MAT-POS-590ac91e-21,type='POSITDockingResults' input_pair=DockingI...,0.05,FNCISWRYQRGLKF-LLVKDONJSA-N,CC(=O)N1CCN(CC1)S(=O)(=O)c2ccc(s2)Cl,0,Mpro-x1402_0A,NaN,-6.284667
0,C[N@](CCO)S(=O)(=O)[N@@]1Cc2ccc(cc2[C@@H](C1)C...,NaN,MAT-POS-5cd9ea36-23,type='POSITDockingResults' input_pair=DockingI...,0.02,RGTAJMPNICITLK-HXUWFJFHSA-N,CC(=O)N1CCN(CC1)S(=O)(=O)c2ccc(s2)Cl,0,Mpro-x1402_0A,NaN,-7.635958
1,C[N@@](CCO)S(=O)(=O)[N@]1Cc2ccc(cc2[C@@H](C1)C...,NaN,MAT-POS-5cd9ea36-23,type='POSITDockingResults' input_pair=DockingI...,0.24,RGTAJMPNICITLK-HXUWFJFHSA-N,Cn1c2c(cncc2NC(=O)Cc3cccc(c3)Cl)cn1,0,Mpro-P1470_0A,NaN,-8.251143
0,Cc1ccc(c(c1F)F)c2cc(cc(c2)Cl)CC(=O)Nc3cnccc3C,NaN,MAT-POS-f42f3716-6,type='POSITDockingResults' input_pair=DockingI...,0.24,KWOZOQHUQWBGBH-UHFFFAOYSA-N,Cn1c2c(cncc2NC(=O)Cc3cccc(c3)Cl)cn1,0,Mpro-P1470_0A,NaN,-8.096560
...,...,...,...,...,...,...,...,...,...,...,...
1,c1ccc(cc1)CC(=O)Nc2cncc3c2cccc3,NaN,RAL-THA-2d450e86-1,type='POSITDockingResults' input_pair=DockingI...,0.24,IBEFUXAVGLACIF-UHFFFAOYSA-N,CC(=O)N1CCN(CC1)S(=O)(=O)c2ccc(s2)Cl,0,Mpro-x1402_0A,NaN,-7.352169
0,C[NH2+][C@@H](c1ccc(c(c1)Cl)Cl)C(=O)Nc2cncc3c2...,NaN,MAT-POS-fce787c2-5,type='POSITDockingResults' input_pair=DockingI...,0.27,TXAXJHHNYJHGRH-KRWDZBQOSA-O,Cn1c2c(cncc2NC(=O)Cc3cccc(c3)Cl)cn1,0,Mpro-P1470_0A,NaN,-7.209109
1,C[NH2+][C@@H](c1ccc(c(c1)Cl)Cl)C(=O)Nc2cncc3c2...,NaN,MAT-POS-fce787c2-5,type='POSITDockingResults' input_pair=DockingI...,0.02,TXAXJHHNYJHGRH-KRWDZBQOSA-O,CC(=O)N1CCN(CC1)S(=O)(=O)c2ccc(s2)Cl,0,Mpro-x1402_0A,NaN,-6.970621
0,c1ccc2c(c1)cncc2N3CC[C@@]4(C3=O)C[N@](Cc5c4cc(...,NaN,EDJ-MED-8bb691af-6,type='POSITDockingResults' input_pair=DockingI...,0.24,ZFIJDHJQBMZSSD-AREMUKBSSA-N,Cn1c2c(cncc2NC(=O)Cc3cccc(c3)Cl)cn1,0,Mpro-P1470_0A,NaN,-7.481847


In [7]:
column_translator = {"SMILES": "Query_SMILES",
                     "ligand_id": "Query_Ligand",
                     "docking-confidence-POSIT": "POSIT_Probability",
                     "complex_ligand_smiles": "Reference_SMILES",
                     "pose_id": "Pose_ID",
                     "docking-structure-POSIT": "Reference_Structure",
                     "docking-score-POSIT":"Chemgauss4"}

In [8]:
updated_df = pose_df[column_translator.keys()].rename(columns=column_translator)

In [9]:
pose_dfm = DataFrameModel(
        name="PoseData",
        type=DataFrameType.POSE,
        dataframe=updated_df,
        key_columns=[
            "Reference_Structure",
            "Query_Ligand",
            "Pose_ID",
        ],
    )

KeyError: "Grouping the dataframe by key_columns: '['Reference_Structure', 'Query_Ligand', 'Pose_ID']'and param_columns: '[]resulted in 4 rows with duplicate values.Perhaps another column is needed!"

In [10]:
updated_df

,Query_SMILES,Query_Ligand,POSIT_Probability,Reference_SMILES,Pose_ID,Reference_Structure,Chemgauss4
0,Cc1ccncc1NC(=O)[C@@H]2CCC3(C2)CC3,MAT-POS-590ac91e-21,0.36,Cn1c2c(cncc2NC(=O)Cc3cccc(c3)Cl)cn1,0,Mpro-P1470_0A,-6.400138
1,Cc1ccncc1NC(=O)[C@@H]2CCC3(C2)CC3,MAT-POS-590ac91e-21,0.05,CC(=O)N1CCN(CC1)S(=O)(=O)c2ccc(s2)Cl,0,Mpro-x1402_0A,-6.284667
0,C[N@](CCO)S(=O)(=O)[N@@]1Cc2ccc(cc2[C@@H](C1)C...,MAT-POS-5cd9ea36-23,0.02,CC(=O)N1CCN(CC1)S(=O)(=O)c2ccc(s2)Cl,0,Mpro-x1402_0A,-7.635958
1,C[N@@](CCO)S(=O)(=O)[N@]1Cc2ccc(cc2[C@@H](C1)C...,MAT-POS-5cd9ea36-23,0.24,Cn1c2c(cncc2NC(=O)Cc3cccc(c3)Cl)cn1,0,Mpro-P1470_0A,-8.251143
0,Cc1ccc(c(c1F)F)c2cc(cc(c2)Cl)CC(=O)Nc3cnccc3C,MAT-POS-f42f3716-6,0.24,Cn1c2c(cncc2NC(=O)Cc3cccc(c3)Cl)cn1,0,Mpro-P1470_0A,-8.096560
...,...,...,...,...,...,...,...
1,c1ccc(cc1)CC(=O)Nc2cncc3c2cccc3,RAL-THA-2d450e86-1,0.24,CC(=O)N1CCN(CC1)S(=O)(=O)c2ccc(s2)Cl,0,Mpro-x1402_0A,-7.352169
0,C[NH2+][C@@H](c1ccc(c(c1)Cl)Cl)C(=O)Nc2cncc3c2...,MAT-POS-fce787c2-5,0.27,Cn1c2c(cncc2NC(=O)Cc3cccc(c3)Cl)cn1,0,Mpro-P1470_0A,-7.209109
1,C[NH2+][C@@H](c1ccc(c(c1)Cl)Cl)C(=O)Nc2cncc3c2...,MAT-POS-fce787c2-5,0.02,CC(=O)N1CCN(CC1)S(=O)(=O)c2ccc(s2)Cl,0,Mpro-x1402_0A,-6.970621
0,c1ccc2c(c1)cncc2N3CC[C@@]4(C3=O)C[N@](Cc5c4cc(...,EDJ-MED-8bb691af-6,0.24,Cn1c2c(cncc2NC(=O)Cc3cccc(c3)Cl)cn1,0,Mpro-P1470_0A,-7.481847


In [11]:
tdf = updated_df.groupby([
            "Reference_Structure",
            "Query_Ligand",
            "Pose_ID",
        ]).count()

In [12]:
(tdf > 1).any(axis=1).sum()

np.int64(4)

In [13]:
tdf[(tdf > 1).any(axis=1)]

Query_SMILES  \
Reference_Structure Query_Ligand       Pose_ID                 
Mpro-P1470_0A       LON-WEI-8f408cad-5 0                   2   
                    TRY-UNI-2eddb1ff-7 0                   2   
Mpro-x1402_0A       LON-WEI-8f408cad-5 0                   2   
                    TRY-UNI-2eddb1ff-7 0                   2   

                                                POSIT_Probability  \
Reference_Structure Query_Ligand       Pose_ID                      
Mpro-P1470_0A       LON-WEI-8f408cad-5 0                        2   
                    TRY-UNI-2eddb1ff-7 0                        2   
Mpro-x1402_0A       LON-WEI-8f408cad-5 0                        2   
                    TRY-UNI-2eddb1ff-7 0                        2   

                                                Reference_SMILES  Chemgauss4  
Reference_Structure Query_Ligand       Pose_ID                                
Mpro-P1470_0A       LON-WEI-8f408cad-5 0                       2           2  
                    TRY-UNI-2eddb1ff-7 0                       2           2  
Mpro-x1402_0A       LON-WEI-8f408cad-5 0                       2           2  
                    TRY-UNI-2eddb1ff-7 0                       2           2

In [14]:
tdf = updated_df.groupby([
            "Reference_Structure",
            "Query_Ligand",
            "Pose_ID",
        ]).nunique()

In [15]:
(tdf > 1).any(axis=1).sum()

np.int64(4)

In [18]:
tdf[(tdf > 1).any(axis=1)].reset_index()

,Reference_Structure,Query_Ligand,Pose_ID,Query_SMILES,POSIT_Probability,Reference_SMILES,Chemgauss4
0,Mpro-P1470_0A,LON-WEI-8f408cad-5,0,2,1,1,2
1,Mpro-P1470_0A,TRY-UNI-2eddb1ff-7,0,2,1,1,2
2,Mpro-x1402_0A,LON-WEI-8f408cad-5,0,2,1,1,2
3,Mpro-x1402_0A,TRY-UNI-2eddb1ff-7,0,2,1,1,2


In [17]:
tdf

Query_SMILES  \
Reference_Structure Query_Ligand        Pose_ID                 
Mpro-P1470_0A       AAR-POS-0daf6b7e-1  0                   1   
                    AAR-POS-0daf6b7e-10 0                   1   
                    AAR-POS-0daf6b7e-14 0                   1   
                    AAR-POS-0daf6b7e-15 0                   1   
                    AAR-POS-0daf6b7e-16 0                   1   
...                                                       ...   
Mpro-x1402_0A       VLA-UCB-29506327-1  0                   1   
                    VLA-UNK-82501c2c-1  0                   1   
                    WAR-XCH-72a8c209-5  0                   1   
                    WAR-XCH-79d12f6e-6  0                   1   
                    WIL-MOD-03b86a88-6  0                   1   

                                                 POSIT_Probability  \
Reference_Structure Query_Ligand        Pose_ID                      
Mpro-P1470_0A       AAR-POS-0daf6b7e-1  0                        1   
                    AAR-POS-0daf6b7e-10 0                        1   
                    AAR-POS-0daf6b7e-14 0                        1   
                    AAR-POS-0daf6b7e-15 0                        1   
                    AAR-POS-0daf6b7e-16 0                        1   
...                                                            ...   
Mpro-x1402_0A       VLA-UCB-29506327-1  0                        1   
                    VLA-UNK-82501c2c-1  0                        1   
                    WAR-XCH-72a8c209-5  0                        1   
                    WAR-XCH-79d12f6e-6  0                        1   
                    WIL-MOD-03b86a88-6  0                        1   

                                                 Reference_SMILES  Chemgauss4  
Reference_Structure Query_Ligand        Pose_ID                                
Mpro-P1470_0A       AAR-POS-0daf6b7e-1  0                       1           1  
                    AAR-POS-0daf6b7e-10 0                       1           1  
                    AAR-POS-0daf6b7e-14 0                       1           1  
                    AAR-POS-0daf6b7e-15 0                       1           1  
                    AAR-POS-0daf6b7e-16 0                       1           1  
...                                                           ...         ...  
Mpro-x1402_0A       VLA-UCB-29506327-1  0                       1           1  
                    VLA-UNK-82501c2c-1  0                       1           1  
                    WAR-XCH-72a8c209-5  0                       1           1  
                    WAR-XCH-79d12f6e-6  0                       1           1  
                    WIL-MOD-03b86a88-6  0                       1           1  

[924 rows x 4 columns]